In [ ]:
from google.colab import drive
from pathlib import Path
import os

# Check if Google Drive is already mounted
if not os.path.exists('/content/drive/MyDrive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("Google Drive is already mounted.")

Google Drive is already mounted.


In [ ]:
# ============================================================
# CELL 1: EXPERIMENT CONFIGURATION
# ============================================================

import json
import random
import unicodedata
from pathlib import Path
from collections import defaultdict
from typing import List, Tuple, Dict, Set, Optional

import pandas as pd
import matplotlib.pyplot as plt


# -----------------------------
# Reproducibility
# -----------------------------

RANDOM_SEED = 42
random.seed(RANDOM_SEED)


# -----------------------------
# Paths
# -----------------------------

# BASE_DIR = Path("/content")  # Change if your project uses another location

BASE_DIR = Path("/content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/Structural_MorphBPE_Experiment")
MODEL_DIR = BASE_DIR / "models"
RESULT_DIR = BASE_DIR / "results"
DATA_PATH = BASE_DIR / "igbo_morph_scoped_corpus.json"

for d in [MODEL_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Environment ready at:", BASE_DIR)


# -----------------------------
# Experimental conditions
# -----------------------------

TARGET_VOCAB_SIZES = [2000, 4000, 8000, 12000, 16000]

TRAIN_RATIO = 0.90
EVAL_SAMPLE_SIZE = 10_000


print("Experiment configuration")
print("-" * 50)
print(f"Random seed:       {RANDOM_SEED}")
print(f"Training ratio:    {TRAIN_RATIO}")
print(f"Evaluation sample: {EVAL_SAMPLE_SIZE}")
print(f"Vocabulary sizes:  {TARGET_VOCAB_SIZES}")
print(f"Data path:         {DATA_PATH}")
print(f"Model directory:   {MODEL_DIR}")
print(f"Results directory: {RESULT_DIR}")

Environment ready at: /content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/Structural_MorphBPE_Experiment
Experiment configuration
--------------------------------------------------
Random seed:       42
Training ratio:    0.9
Evaluation sample: 10000
Vocabulary sizes:  [2000, 4000, 8000, 12000, 16000]
Data path:         /content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/Structural_MorphBPE_Experiment/igbo_morph_scoped_corpus.json
Model directory:   /content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/Structural_MorphBPE_Experiment/models
Results directory: /content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/Structural_MorphBPE_Experiment/results


In [ ]:
# ============================================================
# CELL 2: LOAD MORPHOLOGICALLY ANNOTATED CORPUS
# ============================================================

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_corpus_data = json.load(f)

print("Corpus loaded successfully.")
print(f"Number of word instances: {len(raw_corpus_data):,}")

print("\nExample entry:")
print(raw_corpus_data[0])

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/Structural_MorphBPE_Experiment/igbo_morph_scoped_corpus.json'

In [ ]:
import json
from typing import Optional

WordTuple = Tuple[Tuple[str, ...], ...]

class MorphBPETrainer:
    def __init__(self, special_tokens: Optional[List[str]] = None):
        self.special_tokens = special_tokens or ["<unk>", "<s>", "</s>", "<pad>", "<mask morph>"]
        self.merges: List[Tuple[str, str]] = []
        self.vocab: Dict[str, int] = {}

    def _extract_word_pairs(self, word_tuple: WordTuple) -> List[Tuple[str, str]]:
        pairs = []
        for morpheme in word_tuple:
            for i in range(len(morpheme) - 1):
                pairs.append((morpheme[i], morpheme[i + 1]))
        return pairs

    def _apply_merge_to_morpheme(self, morpheme: Tuple[str, ...], pair: Tuple[str, str]) -> Tuple[str, ...]:
        first, second = pair
        merged = first + second
        new_morpheme = []
        i, n = 0, len(morpheme)
        while i < n:
            if i < n - 1 and morpheme[i] == first and morpheme[i + 1] == second:
                new_morpheme.append(merged)
                i += 2
            else:
                new_morpheme.append(morpheme[i])
                i += 1
        return tuple(new_morpheme)

    def train(self, corpus: Dict[WordTuple, int], target_vocab_size: int):
        self.merges = []
        initial_units = {unit for w in corpus for m in w for unit in m}
        current_vocab = list(self.special_tokens) + sorted(list(initial_units))
        self.vocab = {tok: idx for idx, tok in enumerate(current_vocab)}
        working_corpus = dict(corpus)

        pair_counts: Dict[Tuple[str, str], int] = defaultdict(int)
        where_pair: Dict[Tuple[str, str], Set[WordTuple]] = defaultdict(set)

        for w_tuple, freq in working_corpus.items():
            for p in self._extract_word_pairs(w_tuple):
                pair_counts[p] += freq
                where_pair[p].add(w_tuple)

        while len(self.vocab) < target_vocab_size:
            if not pair_counts:
                break
            best_pair = max(pair_counts, key=pair_counts.get)
            if pair_counts[best_pair] <= 1:
                break

            self.merges.append(best_pair)
            merged_tok = best_pair[0] + best_pair[1]
            if merged_tok not in self.vocab:
                self.vocab[merged_tok] = len(self.vocab)

            words_to_update = list(where_pair[best_pair])
            del where_pair[best_pair], pair_counts[best_pair]

            for old_word in words_to_update:
                if old_word not in working_corpus:
                    continue
                freq = working_corpus.pop(old_word)

                for p in self._extract_word_pairs(old_word):
                    if p != best_pair:
                        pair_counts[p] -= freq
                        if pair_counts[p] <= 0:
                            pair_counts.pop(p, None)
                        where_pair[p].discard(old_word)

                new_word = tuple(self._apply_merge_to_morpheme(m, best_pair) for m in old_word)
                working_corpus[new_word] = working_corpus.get(new_word, 0) + freq

                for p in self._extract_word_pairs(new_word):
                    pair_counts[p] += freq
                    where_pair[p].add(new_word)

    def export_vocab(self, filepath: str):
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(self.vocab, f, ensure_ascii=False, indent=2)

    def export_merges(self, filepath: str):
        with open(filepath, "w", encoding="utf-8") as f:
            f.write("#version: 0.2\n")
            for p in self.merges:
                f.write(f"{p[0]} {p[1]}\n")

class BaselineBPETrainer(MorphBPETrainer):
    def _extract_word_pairs(self, word_tuple: WordTuple) -> List[Tuple[str, str]]:
        pairs = []
        for word_seq in word_tuple:
            for i in range(len(word_seq) - 1):
                pairs.append((word_seq[i], word_seq[i + 1]))
        return pairs

In [ ]:
class BPETokenizer:
    def __init__(self, vocab: Dict[str, int], merges: List[Tuple[str, str]]):
        self.vocab = vocab
        self.merges = merges
        self.merge_ranks = {pair: idx for idx, pair in enumerate(merges)}

    @classmethod
    def from_files(cls, vocab_path: str, merges_path: str) -> "BPETokenizer":
        with open(vocab_path, "r", encoding="utf-8") as f:
            vocab = json.load(f)
        merges = []
        with open(merges_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 2 and not line.startswith("#"):
                    merges.append((parts[0], parts[1]))
        return cls(vocab=vocab, merges=merges)

    def tokenize_word(self, word_str: str) -> List[str]:
        norm_w = normalize_nfc(word_str)
        units = list(unitize_morpheme(norm_w))
        while len(units) > 1:
            best_pair, best_rank = None, float("inf")
            for i in range(len(units) - 1):
                pair = (units[i], units[i + 1])
                rank = self.merge_ranks.get(pair, float("inf"))
                if rank < best_rank:
                    best_rank, best_pair = rank, pair

            if not best_pair or best_rank == float("inf"):
                break

            first, second = best_pair
            new_units, i, n = [], 0, len(units)
            while i < n:
                if i < n - 1 and units[i] == first and units[i + 1] == second:
                    new_units.append(first + second)
                    i += 2
                else:
                    new_units.append(units[i])
                    i += 1
            units = new_units
        return units

def evaluate_corpus(gold_samples: List[List[List[str]]], tokenizer: BPETokenizer) -> Dict:
    total_gold, total_pred, total_hits, total_crossing, total_subwords, total_words = 0, 0, 0, 0, 0, 0
    morpheme_map = defaultdict(list)

    for word_morphemes in gold_samples:
        m_strs = ["".join(c) for c in word_morphemes if c]
        if not m_strs:
            continue

        m_units = [unitize_morpheme(m) for m in m_strs]
        all_units = [u for mu in m_units for u in mu]
        surface = "".join(m_strs)
        pred_toks = tokenizer.tokenize_word(surface)

        total_words += 1
        total_subwords += len(pred_toks)

        gold_b, c_idx = set(), 0
        for idx, mu in enumerate(m_units):
            c_idx += len(mu)
            if idx < len(m_units) - 1:
                gold_b.add(c_idx)
        total_gold += len(gold_b)

        pred_spans, u_ptr = [], 0
        for tok in pred_toks:
            t_len = len(unitize_morpheme(tok))
            pred_spans.append((u_ptr, u_ptr + t_len))
            u_ptr += t_len

        pred_b = {end_u for _, end_u in pred_spans[:-1]}
        total_pred += len(pred_b)
        total_hits += len(gold_b.intersection(pred_b))

        for s_u, e_u in pred_spans:
            if any(s_u < b < e_u for b in gold_b):
                total_crossing += 1

        m_ptr = 0
        for m_str, mu in zip(m_strs, m_units):
            m_s, m_e = m_ptr, m_ptr + len(mu)
            m_ptr = m_e
            m_toks = [tok for (ps, pe), tok in zip(pred_spans, pred_toks) if max(ps, m_s) < min(pe, m_e)]
            morpheme_map[normalize_nfc(m_str)].append(" ".join(m_toks))

    recall = total_hits / total_gold if total_gold > 0 else 1.0
    crossing_rate = total_crossing / total_subwords if total_subwords > 0 else 0.0
    consistency = [
        max(defaultdict(int, {s: seqs.count(s) for s in seqs}).values()) / len(seqs)
        for seqs in morpheme_map.values() if len(seqs) >= 2
    ]
    avg_consistency = sum(consistency) / len(consistency) if consistency else 1.0

    return {
        "boundary_recall": round(recall, 4),
        "crossing_rate": round(crossing_rate, 4),
        "avg_consistency": round(avg_consistency, 4),
        "fertility": round(total_subwords / total_words, 4)
    }

In [ ]:
import random
import time
import matplotlib.pyplot as plt

# NOTE: Ensure the BPETokenizer and evaluate_corpus classes are defined
# This code assumes they are available from a previous cell or import.

# DATA_PATH = BASE_DIR / "igbo_morph_scoped_corpus.json" # Defined in config cell cU8wA90k1bKn
TARGET_VOCAB_SIZES = [2000, 4000, 8000, 12000, 16000]

if not DATA_PATH.exists():
    print(f"Error: The file '{DATA_PATH}' does not exist. Please ensure it is in your Google Drive at the specified location.")
    # Exit or handle the error appropriately if the file is critical
else:
    print(f"Loading data from: {DATA_PATH}")
    with open(DATA_PATH, "r", encoding="utf-8") as f:
        raw_corpus_data = json.load(f)

    random.seed(42)
    shuffled_data = list(raw_corpus_data)
    random.shuffle(shuffled_data)

    split_idx = int(0.9 * len(shuffled_data))
    train_raw = shuffled_data[:split_idx]
    eval_test_sample = shuffled_data[split_idx:][:10000]

    morph_corpus = build_nested_frequency_dict(train_raw)
    baseline_corpus = build_baseline_frequency_dict(train_raw)

    morph_recalls, base_recalls = [], []

    for target_v in TARGET_VOCAB_SIZES:
        print(f"\nTraining Vocab Size: {target_v:,}...")

        # MorphBPE
        morph_trainer = MorphBPETrainer()
        morph_trainer.train(morph_corpus, target_vocab_size=target_v)
        m_v_path, m_m_path = MODEL_DIR / f"m_{target_v}_vocab.json", MODEL_DIR / f"m_{target_v}_merges.txt"
        morph_trainer.export_vocab(str(m_v_path))
        morph_trainer.export_merges(str(m_m_path))
        # Ensure BPETokenizer and evaluate_corpus are available/imported correctly
        # For now, I'll comment out the lines that use them to prevent NameErrors if they aren't yet defined.
        morph_eval = evaluate_corpus(eval_test_sample, BPETokenizer.from_files(str(m_v_path), str(m_m_path)))
        # morph_eval = {"boundary_recall": 0.0} # Placeholder - remove this line as actual evaluation will be done

        # Baseline BPE
        base_trainer = BaselineBPETrainer()
        base_trainer.train(baseline_corpus, target_vocab_size=target_v)
        b_v_path, b_m_path = MODEL_DIR / f"b_{target_v}_vocab.json", MODEL_DIR / f"b_{target_v}_merges.txt"
        base_trainer.export_vocab(str(b_v_path))
        base_trainer.export_merges(str(b_m_path))
        base_eval = evaluate_corpus(eval_test_sample, BPETokenizer.from_files(str(b_v_path), str(b_m_path)))
        # base_eval = {"boundary_recall": 0.0} # Placeholder - remove this line as actual evaluation will be done

        morph_recalls.append(morph_eval["boundary_recall"])
        base_recalls.append(base_eval["boundary_recall"])

        print(f" -> MorphBPE Recall: {morph_eval['boundary_recall']:.4f} | Baseline Recall: {base_eval['boundary_recall']:.4f}")

    # Plot
    plt.figure(figsize=(8, 4.5))
    plt.plot(TARGET_VOCAB_SIZES, morph_recalls, marker="o", color="navy", label="MorphBPE")
    plt.plot(TARGET_VOCAB_SIZES, base_recalls, marker="s", linestyle="--", color="crimson", label="Baseline BPE")
    plt.title("Axis A: Morphological Boundary Recall")
    plt.xlabel("Vocab Size")
    plt.ylabel("Boundary Recall")
    plt.grid(True, linestyle=":", alpha=0.7)
    plt.legend()
    plt.show()

Error: The file '/content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/Structural_MorphBPE_Experiment/igbo_morph_scoped_corpus.json' does not exist. Please ensure it is in your Google Drive at the specified location.


In [ ]:
# Load the trained 16k MorphBPE tokenizer
m_v_path = MODEL_DIR / "m_16000_vocab.json"
m_m_path = MODEL_DIR / "m_16000_merges.txt"

morph_tokenizer = BPETokenizer.from_files(
    str(m_v_path),
    str(m_m_path)
)

print("Vocabulary entries:", len(morph_tokenizer.vocab))
print("Learned merges:", len(morph_tokenizer.merges))

print("\nFirst 20 merges:")
for merge in morph_tokenizer.merges[:20]:
    print(merge)

print("\nLast 20 merges:")
for merge in morph_tokenizer.merges[-20:]:
    print(merge)

Vocabulary entries: 16000
Learned merges: 13937

First 20 merges:
('n', 'a')
('d', 'ị')
('k', 'a')
('m', 'a')
('n', "'")
('b', 'ụ')
('n', 'dị')
('k', 'e')
('r', 'a')
('m', 'e')
('h', 'a')
('r', 'e')
('na', '-')
('t', 'a')
('ny', 'e')
('y', 'a')
('g', 'a')
('l', 'a')
('r', 'i')
('i', 'h')

Last 20 merges:
('at', 's')
('Be', 'at')
('x', 'ed')
('-', '43')
('39', '194')
('39', '4')
('Of', 'fe')
('1.', '2')
('1.', '5')
('Aya', 'nna')
('lap', 'to')
('Ari', 'nze')
('Ari', 'zo')
('Ari', 'bo')
('ọba', 'nte')
('họ', 'tel')
('lin', 'tel')
('Ch', 'iwe')
('Ibi', 'bio')
('Ndu', 'lue')


In [ ]:
# Inspect one evaluation example containing multiple morphemes
for sample in eval_test_sample:
    if len(sample) > 1:
        gold_morphemes = ["".join(m) for m in sample]
        surface = "".join(gold_morphemes)

        print("GOLD MORPHEMES:", gold_morphemes)
        print("SURFACE:", surface)
        print("TOKENS:", morph_tokenizer.tokenize_word(surface))

        break

GOLD MORPHEMES: ['ga-e', 'nye']
SURFACE: ga-enye
TOKENS: ['ga-e', 'nye']


In [ ]:
morph_eval = evaluate_corpus(
    eval_test_sample,
    morph_tokenizer
)

print(morph_eval)

{'boundary_recall': 0.6433, 'crossing_rate': 0.0816, 'avg_consistency': 0.9346, 'fertility': 1.4718}


In [ ]:
b_v_path = MODEL_DIR / "b_16000_vocab.json"
b_m_path = MODEL_DIR / "b_16000_merges.txt"

base_tokenizer = BPETokenizer.from_files(
    str(b_v_path),
    str(b_m_path)
)

base_eval = evaluate_corpus(
    eval_test_sample,
    base_tokenizer
)

print(base_eval)

{'boundary_recall': 0.0808, 'crossing_rate': 0.1761, 'avg_consistency': 0.8294, 'fertility': 1.2803}
